In [58]:
from pathlib import Path

import torch
from torch import nn
from torchvision import datasets,transforms
from torch.utils.data import Subset,random_split,DataLoader

In [59]:
data_path = Path('data/')
data_path

WindowsPath('data')

In [60]:
list(data_path.iterdir())

[WindowsPath('data/Tomato_Bacterial_spot'),
 WindowsPath('data/Tomato_Early_blight'),
 WindowsPath('data/Tomato_healthy'),
 WindowsPath('data/Tomato_Late_blight'),
 WindowsPath('data/Tomato_Leaf_Mold'),
 WindowsPath('data/Tomato_Septoria_leaf_spot'),
 WindowsPath('data/Tomato_Spider_mites_Two_spotted_spider_mite'),
 WindowsPath('data/Tomato__Target_Spot'),
 WindowsPath('data/Tomato__Tomato_mosaic_virus'),
 WindowsPath('data/Tomato__Tomato_YellowLeaf__Curl_Virus')]

In [61]:
train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

In [62]:
plant_dataset = datasets.ImageFolder(root=data_path)

print(len(plant_dataset))
print(plant_dataset.class_to_idx)

16011
{'Tomato_Bacterial_spot': 0, 'Tomato_Early_blight': 1, 'Tomato_Late_blight': 2, 'Tomato_Leaf_Mold': 3, 'Tomato_Septoria_leaf_spot': 4, 'Tomato_Spider_mites_Two_spotted_spider_mite': 5, 'Tomato__Target_Spot': 6, 'Tomato__Tomato_YellowLeaf__Curl_Virus': 7, 'Tomato__Tomato_mosaic_virus': 8, 'Tomato_healthy': 9}


In [63]:
torch.manual_seed(42)

train_size = int(0.7*len(plant_dataset))
val_size = int(0.15*len(plant_dataset))
test_size = len(plant_dataset) - train_size - val_size

train_subset,val_subset,test_subset = random_split(
    plant_dataset,
    [train_size,val_size,test_size]
)


In [64]:
train_dataset = datasets.ImageFolder(root=data_path,
                                     transform=train_transform)

val_dataset = datasets.ImageFolder(root=data_path,
                                   transform=test_transform)

test_dataset = datasets.ImageFolder(root=data_path,
                                    transform=test_transform)

In [65]:
train_dataset = Subset(train_dataset,train_subset.indices)
val_dataset = Subset(val_dataset,val_subset.indices)
test_dataset = Subset(test_dataset,test_subset.indices)

In [66]:
train_loader = DataLoader(dataset=train_dataset,
                          batch_size=32,
                          shuffle=True)

val_loader = DataLoader(dataset=val_dataset,
                        batch_size=32,
                        shuffle=False)

test_loader = DataLoader(dataset=test_dataset,
                         batch_size=32,
                         shuffle=False)

In [67]:
class Baseline(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3,16,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*16*16,128),
            nn.ReLU(),
            nn.Linear(128,len(plant_dataset.classes))
        )

    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)

        return x

torch.manual_seed(42)

basemodel = Baseline()
basemodel

Baseline(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=16384, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [68]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(params=basemodel.parameters(),lr=0.001)

In [69]:
def train_step(model,dataloader,loss_fn,optimizer):
    model.train()

    train_loss = 0
    correct = 0
    total = 0

    for images,labels in dataloader:

        output = model(images)
        loss = loss_fn(output,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        preds = output.argmax(dim=1)
        correct += (preds==labels).sum().item()
        total += labels.size(0)

    train_loss /= len(dataloader)
    train_acc = (correct/total)*100

    return train_loss,train_acc


In [70]:
def val_step(model,dataloader,loss_fn):
    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.inference_mode():
        for images,labels in dataloader:
            output = model(images)
            loss = loss_fn(output,labels)

            val_loss += loss.item()

            preds = output.argmax(dim=1)
            correct += (preds==labels).sum().item()
            total += labels.size(0)

    val_loss /= len(dataloader)
    val_acc = (correct/total)*100

    return val_loss,val_acc      

In [71]:
epochs = 5 

for epoch in range(epochs):
    train_loss,train_acc = train_step(basemodel,train_loader,loss_fn,optimizer)
    val_loss,val_acc = val_step(basemodel,val_loader,loss_fn)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )


Epoch 1/5 | Train Loss: 1.0281 | Train Acc: 65.1914 | Val Loss: 0.5989 | Val Acc: 79.8417
Epoch 2/5 | Train Loss: 0.4864 | Train Acc: 83.3408 | Val Loss: 0.6262 | Val Acc: 79.1337
Epoch 3/5 | Train Loss: 0.3401 | Train Acc: 88.3376 | Val Loss: 0.3466 | Val Acc: 87.6302
Epoch 4/5 | Train Loss: 0.2801 | Train Acc: 89.9438 | Val Loss: 0.4860 | Val Acc: 81.9242
Epoch 5/5 | Train Loss: 0.2305 | Train Acc: 91.6748 | Val Loss: 0.2632 | Val Acc: 91.0454


In [72]:
def test_step(model,dataloader,loss_fn):
    model.eval()

    test_loss = 0
    correct = 0
    total = 0

    with torch.inference_mode():
        for images,labels in dataloader:

            outputs = model(images)
            loss = loss_fn(outputs,labels)

            test_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    test_loss /= len(dataloader)
    test_acc = (correct/total)*100

    return test_loss,test_acc

In [74]:
test_loss,test_acc = test_step(model=basemodel,
                               dataloader=test_loader,
                               loss_fn=loss_fn)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

Test Loss: 0.2840
Test Accuracy: 91.8019
